In [55]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import torchvision.transforms.functional as F

# 1. Setup Device (M2 Chip Support)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

class SquarePad:
    def __call__(self, image):
        w, h = image.size
        max_wh = max(w, h)
        hp = (max_wh - w) // 2
        vp = (max_wh - h) // 2
        padding = (hp, vp, max_wh - w - hp, max_wh - h - vp)
        return F.pad(image, padding, (200,200,200), 'constant')



# 2. Online Augmentation Pipeline
data_transforms = transforms.Compose([
    SquarePad(), # First, make it a square with padding
    transforms.Resize((224, 224)), # Standard size for pre-trained models
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    # transforms.ColorJitter(brightness=0.2, contrast=0.2), # Handles lighting variance
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # Standard ImageNet values
])

Using device: mps


In [56]:
# 3. Load Pre-trained Model
# We use weights from ImageNet (millions of general images)
model = models.resnet18(weights='IMAGENET1K_V1')

# 4. "Freeze" the Backbone
# This prevents the model from forgetting basic shapes/textures
for param in model.parameters():
    param.requires_grad = False

# 5. Replace the "Head"
# We change the last layer to output 2 classes (e.g., Tripos vs. Other)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2) 

model = model.to(device)

# 6. Loss and Optimizer (tripos is 2x more importanat)
class_weights = torch.tensor([1.0, 2.0]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
# Only optimize the parameters of the final layer (the head)
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

In [57]:
import os
from torchvision.datasets import ImageFolder

# Define the path to your folders
data_dir = './data' 

# Load the datasets
train_dataset = ImageFolder(root=os.path.join(data_dir, 'train'), transform=data_transforms)
val_dataset = ImageFolder(root=os.path.join(data_dir, 'val'), transform=data_transforms)

# Create DataLoaders (this feeds the images into your M2 GPU in batches)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

print(f"Classes found: {train_dataset.classes}") # Should output ['other', 'tripos']
print(f"Folder Mapping: {train_dataset.class_to_idx}")

Classes found: ['other', 'tripos']
Folder Mapping: {'other': 0, 'tripos': 1}


In [58]:
import time
import copy

def train_phytoplankton_model(model, criterion, optimizer, num_epochs=10):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
                dataloader = train_loader
            else:
                model.eval()   # Set model to evaluate mode
                dataloader = val_loader

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data
            for inputs, labels in dataloader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # Zero the parameter gradients
                optimizer.zero_grad()

                # Forward pass
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # Backward pass + optimize
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # Statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(dataloader.dataset)
            epoch_acc = running_corrects.float() / len(dataloader.dataset)

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # Deep copy the model if it's the most accurate one so far
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:4f}')

    # Load best model weights
    model.load_state_dict(best_model_wts)
    return model



In [59]:
# To start training:
model = train_phytoplankton_model(model, criterion, optimizer, num_epochs=40)

Epoch 0/39
----------
train Loss: 0.6381 Acc: 0.5463
val Loss: 0.5975 Acc: 0.7200

Epoch 1/39
----------
train Loss: 0.6284 Acc: 0.6019
val Loss: 0.7750 Acc: 0.4400

Epoch 2/39
----------
train Loss: 0.6090 Acc: 0.5926
val Loss: 0.5779 Acc: 0.7600

Epoch 3/39
----------
train Loss: 0.5892 Acc: 0.6019
val Loss: 0.7172 Acc: 0.6000

Epoch 4/39
----------
train Loss: 0.5693 Acc: 0.6667
val Loss: 0.5173 Acc: 0.7600

Epoch 5/39
----------
train Loss: 0.5876 Acc: 0.6111
val Loss: 0.4771 Acc: 0.7600

Epoch 6/39
----------
train Loss: 0.5441 Acc: 0.6759
val Loss: 0.8688 Acc: 0.3200

Epoch 7/39
----------
train Loss: 0.4839 Acc: 0.6389
val Loss: 0.5268 Acc: 0.8000

Epoch 8/39
----------
train Loss: 0.4718 Acc: 0.7500
val Loss: 0.6772 Acc: 0.5600

Epoch 9/39
----------
train Loss: 0.4478 Acc: 0.7037
val Loss: 0.6167 Acc: 0.6400

Epoch 10/39
----------
train Loss: 0.4187 Acc: 0.7778
val Loss: 0.5159 Acc: 0.7600

Epoch 11/39
----------
train Loss: 0.4502 Acc: 0.7593
val Loss: 0.4956 Acc: 0.7600

Ep

In [60]:
torch.save(model.state_dict(), 'tripos_classifier_v14.pth')